## Magnetostatic problem


In [627]:
from ngsolve import *
from netgen.read_gmsh import ReadGmsh
from ngsolve.webgui import Draw
from netgen.csg import *
import math
import pyvista as pv
import numpy as np
from ngsolve.krylovspace import GMRes

mesh_path = '../../meshes/coil_box_named'
output_path = '../../output/case2/case2_ngsolve'

# Import geometries
mesh = ReadGmsh(mesh_path + ".msh")

# for i in range(1, 3):
#     # print(i)
#     mesh.SetMaterial(i, f'{i}')

# for i in range(1, 13):
#     # print(i)
#     mesh.SetBCName(i-1, f'{i}')

mesh = Mesh(mesh)

mesh.ngmesh.Save(mesh_path + ".vol")

In [628]:
mesh.ne, mesh.nv, mesh.GetMaterials(), mesh.GetBoundaries()

(69629, 12090, ('vacuum', 'wire'), ('CoilIn', 'CoilOut'))

In [629]:
# Define material, coil, and BC parameters

I_coil = 2191.9 # Input electric current [A]
sigma_coil = 62.83185 # Coil's electric conductivity [S/m]
mu_r_coil = 1.0 # Coil's relative permeability [-]
mu0 = 4*math.pi*1e-7

# mu0 = 1.0

print(1/mu0)

795774.7154594767


In [630]:
sigma = {"vacuum": 0.0, "wire": sigma_coil}  # Electric conductivity [S/m]
mu_r = {"vacuum": 1.0, "wire": mu_r_coil} # Relative permeability [-]

sigma_cf = CoefficientFunction([sigma.get(mat, 0.0) for mat in mesh.GetMaterials()])
mu_cf = mu0 * CoefficientFunction([mu_r.get(mat, 1.0) for mat in mesh.GetMaterials()])

crosssection = Integrate(1, mesh, definedon=mesh.Boundaries("CoilIn"))

print(f"Coil cross section = {crosssection} m^2.")

r = np.sqrt(crosssection / math.pi)

print(f"Coil radius = {r} m.")

A_nom = mu0 * I_coil / (4 * math.pi * r)

print(f"Scale of A = {A_nom} Wb/m.")

fespot = H1(mesh, order=1, definedon=mesh.Materials("wire"), dirichlet="CoilOut")
phi,psi = fespot.TnT()
with TaskManager():
    bfa = BilinearForm(sigma_cf*grad(phi)*grad(psi)*dx).Assemble()
    inv = bfa.mat.Inverse(freedofs=fespot.FreeDofs(), inverse="sparsecholesky")
    lff = LinearForm(I_coil/crosssection*psi*ds("CoilIn")).Assemble()
    gfphi = GridFunction(fespot)
    gfphi.vec.data = inv * lff.vec

gfcurrdens = -sigma_cf*grad(gfphi)

Coil cross section = 7.0710678118656e-05 m^2.
Coil radius = 0.004744249983287986 m.
Scale of A = 0.04620119107806607 Wb/m.


In [631]:
# fes = HCurl(mesh, order=1, complex=True, dirichlet="VacuumSurface", nograds = False)

# fes = HCurl(mesh, order=1, nograds=True)

# fes = HCurl(mesh, order=1, gradientdomains="wire")

# fes = HCurl(mesh, order=1, nograds=True)

# fes = HCurl(mesh, order=1, dirichlet="Side1|Side2|Side3|Side4|Side5|Side6", nograds=True)


fes = HCurl(mesh, order=1, dirichlet="1|2|3|4|5|12", nograds=True) # THIS ONE

# fes = HCurl(mesh, order=1, dirichlet="1|2|3|4|5|12", nograds=False)

print ("HCurl dofs:", fes.ndof)
u,v = fes.TnT()
a = BilinearForm(1/mu_cf*curl(u)*curl(v)*dx+1e-7/mu_cf*u*v*dx)
# pre = preconditioners.BDDC(a)
# pre = preconditioners.HCurlAMG(a)
# pre = preconditioners.MultiGrid(a)
# pre = BilinearForm(u*v*ds, diagonal=True).Assemble().mat.Inverse()
pre = preconditioners.Local(a)
f = LinearForm(sigma_cf*grad(gfphi)*v*dx("wire"))
with TaskManager():
    a.Assemble()
    f.Assemble()

HCurl dofs: 82609


In [632]:
A = GridFunction(fes)
A.vec[:] = 0
# pre = preconditioners.Local(a)
A.vec.data = GMRes(a.mat, f.vec, pre=pre, maxsteps=100, printrates=True, tol= 1e-9)

# A.vec.data = GMRes(a.mat, f.vec, freedofs=fes.FreeDofs(), maxsteps=100, printrates=True, tol= 1e-9)

# inv = CGSolver(a.mat, pre, precision=1e-9, maxsteps=100, printrates=True)
# A.vec.data = inv * f.vec

# A.vec.data = solvers.CG(a.mat, f.vec, pre=pre, tol= 1e-9)

# solvers.BVP(bf=a, lf=f, gf=A, pre=pre, \
#             solver=solvers.CGSolver, solver_flags={"plotrates": True, "tol" : 1e-12})

# solvers.BVP(bf=a, lf=f, gf=A, pre=pre, \
#             solver=solvers.CGSolver, solver_flags={"plotrates": True, "tol" : 1e-12})

GMRes iteration 1, residual = 4.4583171324915095e-06     
GMRes iteration 2, residual = 3.932137566049639e-06     
GMRes iteration 3, residual = 3.506955899641236e-06     
GMRes iteration 4, residual = 3.0578299196849756e-06     
GMRes iteration 5, residual = 2.7384775459209636e-06     
GMRes iteration 6, residual = 2.5350149794011905e-06     
GMRes iteration 7, residual = 2.402723915197799e-06     
GMRes iteration 8, residual = 2.3031029369168713e-06     
GMRes iteration 9, residual = 2.229803196464924e-06     
GMRes iteration 10, residual = 2.177717943784483e-06     
GMRes iteration 11, residual = 2.136469494709124e-06     
GMRes iteration 12, residual = 2.101428877998619e-06     
GMRes iteration 13, residual = 2.069403678683037e-06     
GMRes iteration 14, residual = 2.04002782221802e-06     
GMRes iteration 15, residual = 2.0143433198363583e-06     
GMRes iteration 16, residual = 1.991958656965241e-06     
GMRes iteration 17, residual = 1.971675183656981e-06     
GMRes iteration 18

In [633]:
fes_A = VectorH1(mesh, order=3)
A_gf = GridFunction(fes_A)
A_gf.Set(A.real)

In [634]:
B = curl(A)
# fes_B = VectorH1(mesh, order=3)
# B_gf = GridFunction(fes_B)
# B_gf.Set(B.real)

In [635]:
# fespot_global = H1(mesh, order=1)
# gfphi_global = GridFunction(fespot_global)
# gfphi_global.Set(gfphi, definedon=mesh.Materials("wire"))

# fescurrden_global = VectorH1(mesh, order=1)
# # fescurrden_global = VectorL2(mesh, order=0)
# # fescurrden_global = HCurl(mesh, order=1)
# # fescurrden_global = HDiv(mesh, order=1)
# gfcurrdens_global = GridFunction(fescurrden_global)
# gfcurrdens_global.Set(gfcurrdens, definedon=mesh.Materials("wire"))

In [636]:
# vtk = VTKOutput(mesh,coefs=[gfphi],names=["sol"],filename=output_path + "electric_potential",subdivision=0)
# vtk.Do()

# res = pv.read(mesh_path + ".msh")
# points = res.points
# gfphi_out = np.zeros((points.shape[0], 1))
# gfcurrdens_out = np.zeros((points.shape[0], 3))
# B_sol = np.zeros((points.shape[0], 3))
# A_sol = np.zeros((points.shape[0], 3))

# for i in range(points.shape[0]):
#     point = mesh(points[i, 0], points[i, 1], points[i, 2])
#     gfphi_out[i, :] = gfphi_global(point)
#     B_sol[i, :] = B_gf(point)
#     A_sol[i, :] = A_gf(point)
#     gfcurrdens_out[i, :] = gfcurrdens_global(point)

# res["electric_potential"] = gfphi_out
# res["magnetic_flux_density"] = B_sol
# res["magnetic_vector_potential"] = A_sol
# res["current_density"] = gfcurrdens_out

# res.save(output_path + ".vtu")



vtk = VTKOutput(mesh,coefs=[A, B, gfcurrdens, A_gf, gfphi],
                names=["magnetic_vector_potential_nd", "magnetic_flux_density", "current_density", "magnetic_vector_potential", "electric_potential"],
                filename=output_path, subdivision=0)
vtk.Do()

'../../output/case2/case2_ngsolve'